# Google Places API — Rating & Review Extraction

Connects to the Google Places API to retrieve overall ratings, price levels, and up to 5 review texts per business.

**Input:** `Data/Quer_file_03_06_26.xlsx`  
**Output:** `Data/Quer_file_output_03_06_26.xlsx`

> **Note:** Requires a valid Google Places API key. Set `API_KEY` below before running.

In [ ]:
import pandas as pd
import requests
import time
import os
from pathlib import Path

# ── Resolve repo root ──────────────────────────────────────────────────────
if '__file__' in dir():
    REPO_ROOT = Path(__file__).resolve().parent.parent
else:
    REPO_ROOT = Path().resolve()
os.chdir(REPO_ROOT)
DATA_DIR = REPO_ROOT / "Data"

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")


## 1. Configuration

In [ ]:
# ── Set your API key here ──────────────────────────────────────────────────
API_KEY = "YOUR_GOOGLE_PLACES_API_KEY"   # <-- replace before running

INPUT_FILE  = DATA_DIR / "Quer_file_03_06_26.xlsx"
OUTPUT_FILE = DATA_DIR / "Quer_file_output_03_06_26.xlsx"


## 2. Load Input Data

In [ ]:
%%time
df = pd.read_excel(INPUT_FILE)
print(f"Loaded {len(df):,} businesses")
print(df[['BusinessName','PostCode']].head())


## 3. API Fetch Function

In [ ]:
def get_google_place_data(name, postcode):
    """
    Fetches rating, price level, and up to 5 review texts for a business.
    Returns (overall_rating, price_level, reviews_combined, review_ratings_combined).
    """
    query = f"{name} {postcode}"
    search_url = (
        f"https://maps.googleapis.com/maps/api/place/textsearch/json"
        f"?query={query}&key={API_KEY}"
    )
    try:
        search_resp = requests.get(search_url, timeout=10).json()
        if search_resp.get('status') == 'OK' and search_resp.get('results'):
            place_id = search_resp['results'][0]['place_id']
            details_url = (
                f"https://maps.googleapis.com/maps/api/place/details/json"
                f"?place_id={place_id}&fields=rating,price_level,reviews&key={API_KEY}"
            )
            details_resp = requests.get(details_url, timeout=10).json()
            if details_resp.get('status') == 'OK':
                res = details_resp.get('result', {})
                overall_rating = res.get('rating', 'N/A')
                price_level    = res.get('price_level', 'N/A')
                reviews_list   = res.get('reviews', [])
                texts   = [r.get('text','').replace('\n',' ') for r in reviews_list]
                ratings = [str(r.get('rating','')) for r in reviews_list]
                return (
                    overall_rating,
                    price_level,
                    ' | '.join(texts)   or 'N/A',
                    ' | '.join(ratings) or 'N/A',
                )
    except Exception as e:
        print(f"  Error for {name}: {e}")
    return 'N/A', 'N/A', 'N/A', 'N/A'


## 4. Batch Fetch with Progress Logging

In [ ]:
%%time
overall_ratings, price_levels, all_reviews, all_review_ratings = [], [], [], []

print(f"Fetching data for {len(df):,} businesses...")
for idx, row in df.iterrows():
    name, postcode = row.get('BusinessName'), row.get('PostCode')
    if pd.isna(name) or pd.isna(postcode):
        overall_ratings.append('N/A'); price_levels.append('N/A')
        all_reviews.append('N/A');    all_review_ratings.append('N/A')
        continue
    if idx % 500 == 0:
        print(f"  {idx:,} / {len(df):,} processed...")
    r, p, t, rv = get_google_place_data(name, postcode)
    overall_ratings.append(r); price_levels.append(p)
    all_reviews.append(t);     all_review_ratings.append(rv)
    time.sleep(0.2)

df['Google_Overall_Rating']          = overall_ratings
df['Google_Price_Level']             = price_levels
df['Google_Review_Texts']            = all_reviews
df['Google_Individual_Review_Ratings'] = all_review_ratings

print(f"\nFetch complete.")
print(f"  Businesses with ratings : {(df['Google_Overall_Rating'] != 'N/A').sum():,}")
print(f"  Businesses without      : {(df['Google_Overall_Rating'] == 'N/A').sum():,}")


## 5. Save Output

In [ ]:
df.to_excel(OUTPUT_FILE, index=False)
print(f"✓ Saved {len(df):,} rows to {OUTPUT_FILE}")
